In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, '../src')
np.random.seed(42)

from data_loader import DataLoader
from model_trainer import ModelTrainer, ModelEvaluator
from drift_detectors import MultiWindowDriftAnalysis

# Prepare data
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

loader = DataLoader(random_state=42)
df_synthetic, _ = loader.load_synthetic_data(
    n_samples_per_window=config['dataset']['synthetic']['n_samples_per_window'],
    n_features=config['dataset']['synthetic']['n_features'],
    n_windows=config['dataset']['synthetic']['n_windows'],
    drift_magnitude=config['dataset']['synthetic']['drift_magnitude'],
    drift_type=config['dataset']['synthetic']['drift_type']
)

windows = loader.split_into_windows(df_synthetic, n_windows=config['time_window']['n_windows'])
splits = loader.get_ref_and_test_splits(windows, ref_window_idx=0)
splits = loader.standardize_splits(splits)

print(f"✓ Data prepared: {len(splits)} windows")


In [ ]:
# Train baseline models on reference window
model_types = config['model']['types']
trained_models = {}

ref_split = splits[0]
X_ref = ref_split['X_ref']
y_ref = ref_split['y_ref']

for model_type in model_types:
    print(f"\nTraining {model_type}...")
    model = ModelTrainer(model_type=model_type, random_state=42)
    model.train(X_ref, y_ref)
    trained_models[model_type] = model
    print(f"✓ {model_type} trained on {X_ref.shape[0]} samples")


In [ ]:
# Evaluate models on all windows
evaluator = ModelEvaluator(metrics=['accuracy', 'f1', 'auc_roc', 'brier_score'])

all_results = {}
for model_type, model in trained_models.items():
    print(f"\nEvaluating {model_type}...")
    perf_df = evaluator.evaluate_all_windows(model, splits)
    all_results[model_type] = perf_df
    print(f"✓ Evaluation complete for {model_type}")
    print(f"\nPerformance summary:")
    print(perf_df[['window', 'accuracy', 'auc_roc', 'f1']].head(10))


In [ ]:
# Compute performance degradation
degradation_results = {}
for model_type, perf_df in all_results.items():
    degradation = evaluator.compute_performance_degradation(perf_df, ref_window_idx=0)
    degradation_results[model_type] = degradation
    
    print(f"\n{model_type} - Performance Degradation:")
    for metric, values in degradation.items():
        max_deg = values.max()
        mean_deg = values.mean()
        print(f"  {metric}: max={max_deg:.4f}, mean={mean_deg:.4f}")


In [ ]:
from visualizer import DriftVisualizer

visualizer = DriftVisualizer(output_dir='../reports/figures')

# Plot performance over time for all models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['accuracy', 'f1', 'auc_roc', 'brier_score']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    for model_type, perf_df in all_results.items():
        if metric in perf_df.columns:
            ax.plot(perf_df['window'], perf_df[metric], marker='o', label=model_type, linewidth=2)
    
    ax.set_xlabel("Time Window")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} Over Time", fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Model Performance Degradation Over Time (No Retraining)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_performance_degradation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Performance degradation plot saved")


## 5. Summary: Baseline Model Performance

**Key Findings:**
- All models show some performance degradation over time without retraining
- The magnitude of degradation varies by model type
- This degradation is likely driven by the drift in feature distributions
- Next: Test if retraining strategies can mitigate this degradation


## 4. Visualize Performance Over Time


## 3. Compute Performance Degradation


## 2. Evaluate Models on All Time Windows


## 1. Train Baseline Models


# 03_model_degradation: Training Models and Analyzing Performance Degradation

This notebook trains baseline models on the reference window and tracks their performance degradation over time windows.
